In [ ]:
# import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import cartopy.crs as ccrs
import cartopy.feature as cf
import imageio
import matplotlib.colors as mcolors
from tqdm import trange
import os

# Define file paths
csv_base_path = "G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\LCMAP_edges\\Ecoregion_Classification\\county_{}.csv"
forest_data_base_path = "G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Forest_Depth_Classification\\Ecoregion_Classification\\county_{}.csv"
shapefile_path = "G:\\Hangkai\\CONUS Forest Edge Mapping\\CONUS shapefile\\cb_2018_us_county_5m.shp"
county_shapefile = gpd.read_file(shapefile_path)

# Ensure the county shapefile 'STATEFP' and 'COUNTYFP' are strings for consistent merging
county_shapefile['STATEFP'] = county_shapefile['STATEFP'].astype(str).str.zfill(2)
county_shapefile['COUNTYFP'] = county_shapefile['COUNTYFP'].astype(str).str.zfill(3)

# Edge lengths for each code from 1 to 15
edge_lengths_dict = {
    1: 1, 2: 1, 3: 2, 4: 1, 5: 2, 6: 2, 7: 3, 8: 1, 9: 2,
    10: 2, 11: 3, 12: 2, 13: 3, 14: 3, 15: 4
}

# Initialize regression parameters
slope = None
intercept = None

# Prepare a folder to save the images
output_dir = 'Area_Edge'
os.makedirs(output_dir, exist_ok=True)

def process_data(year):
    edge_csv_path = csv_base_path.format(year)
    forest_csv_path = forest_data_base_path.format(year)
    
    edge_data = pd.read_csv(edge_csv_path)
    forest_data = pd.read_csv(forest_csv_path)

    # Standardize key columns
    edge_data['STATEFP'] = edge_data['STATEFP'].astype(str).str.zfill(2)
    edge_data['COUNTYFP'] = edge_data['COUNTYFP'].astype(str).str.zfill(3)
    forest_data['STATEFP'] = forest_data['STATEFP'].astype(str).str.zfill(2)
    forest_data['COUNTYFP'] = forest_data['COUNTYFP'].astype(str).str.zfill(3)

    # Calculate total edge length and forest area
    edge_data['total_edge_length'] = edge_data[[str(i) for i in range(1, 16)]].fillna(0).apply(lambda x: np.dot(x, [0.03 * i for i in range(1, 16)]), axis=1)
    forest_data['total_forest_area'] = forest_data[[str(i) for i in range(1, 6)]].sum(axis=1) * 0.03 * 0.03

    # Filter out counties with zero or NaN forest area
    forest_data = forest_data[forest_data['total_forest_area'] > 0]

    # Merge data frames
    combined_data = edge_data.merge(forest_data[['STATEFP', 'COUNTYFP', 'total_forest_area']], on=['STATEFP', 'COUNTYFP'], how='inner')
    combined_data['log_forest_area'] = np.log(combined_data['total_forest_area'])
    combined_data['log_edge_length'] = np.log(combined_data['total_edge_length'])

    return combined_data


# Calculate regression baseline from 1985
data_1985 = process_data(1985)
slope, intercept, _, _, _ = stats.linregress(data_1985['log_forest_area'], data_1985['log_edge_length'])

# Function to plot data and save as image
def plot_data(year, data):
    data['estimated_log_edge'] = intercept + slope * data['log_forest_area']
    data['log_residuals'] = data['log_edge_length'] - data['estimated_log_edge']

    # Merge with shapefile and handle NaN residuals
    merged_shapefile = county_shapefile.merge(data[['STATEFP', 'COUNTYFP', 'log_residuals']], on=['STATEFP', 'COUNTYFP'], how='left')
    merged_shapefile['log_residuals'] = merged_shapefile['log_residuals'].fillna(value=np.nan)  # Ensure NaNs are handled if any edge cases are missed

    # Plot
    norm = mcolors.Normalize(vmin=-1, vmax=1)
    cmap = plt.get_cmap('coolwarm').copy()  # Copy the colormap to modify it
    cmap.set_bad('black')  # Set NaN values to black

    fig, ax = plt.subplots(1, 1, figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    merged_shapefile.plot(norm=norm, column='log_residuals', ax=ax, cmap=cmap, linewidth=0.8, edgecolor='k',missing_kwds={'color': 'black', 'edgecolor': 'black'})
    ax.set_extent([-125, -67, 25, 50], crs=ccrs.PlateCarree())
    # Color bar setup
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.036, pad=0.1, label='Log Residuals')
    
    ax.set_title(f'Log Residuals of Edge Length vs. Forest Area ({year})')
    filepath = os.path.join(output_dir, f'Area_Edge_{year}.png')
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()

    return filepath

# Create GIF
image_paths = [plot_data(year, process_data(year)) for year in trange(1985, 2022)]
images = [imageio.imread(path) for path in image_paths]
imageio.mimsave(os.path.join('log_residuals_animation_v1.gif'), images, fps=5)

print("GIF created successfully!")


In [ ]:
print(stats.linregress(data_1985['log_forest_area'], data_1985['log_edge_length']))
import matplotlib.pyplot as plt

plt.scatter(data_1985['log_forest_area'], data_1985['log_edge_length'])
plt.xlabel('Forest Area')
plt.ylabel('Edge Length')
plt.grid(True)
plt.show()

print(stats.linregress(data_1985['total_forest_area'], data_1985['total_edge_length']))
plt.scatter(data_1985['total_forest_area'], data_1985['total_edge_length'])
plt.xlabel('Forest Area')
plt.ylabel('Edge Length')
plt.grid(True)
plt.show()